# 1. Feature Extraction & Dataset Preparation
This notebook downloads the **MSR-VTT** dataset from Hugging Face, loads the pre-trained multimodal encoders, extracts visual, audio, and video teacher features, and caches them for model training.


## Step 1: Install remote dependencies
We install the required libraries directly on the Kaggle GPU instance.


In [1]:
!pip install -q git+https://github.com/facebookresearch/ImageBind.git
!pip install -q git+https://github.com/openai/CLIP.git
!pip install -q decord pytorchvideo resampy soundfile tqdm


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 91.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 57.4 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires num

## Step 2: Import libraries & Verify GPU


In [2]:
import os
import zipfile
import urllib.request
import json
import torch
import numpy as np
import PIL.Image as Image
from tqdm import tqdm
import torchaudio
import decord
from decord import VideoReader, cpu

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Decord GPU support: {decord.__file__}")


Using device: cuda
Decord GPU support: /usr/local/lib/python3.12/dist-packages/decord/__init__.py


## Step 3: Download and Extract MSR-VTT (Hugging Face Mirror)
We download a standard 1K test split and 7K training split.


In [3]:
os.makedirs("msrvtt", exist_ok=True)

# Download files
urls = {
    "msrvtt/msrvtt_train_7k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_train_7k.json",
    "msrvtt/msrvtt_test_1k.json": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/msrvtt_test_1k.json",
    "msrvtt/MSRVTT_Videos.zip": "https://huggingface.co/datasets/friedrichor/MSR-VTT/resolve/main/MSRVTT_Videos.zip"
}

for path, url in urls.items():
    if not os.path.exists(path):
        print(f"Downloading {path}...")
        urllib.request.urlretrieve(url, path)
        print("Done.")

# Unzip videos
video_dir = "msrvtt/MSRVTT_Videos"
if not os.path.exists(video_dir):
    print("Extracting videos...")
    with zipfile.ZipFile("msrvtt/MSRVTT_Videos.zip", 'r') as zip_ref:
        zip_ref.extractall("msrvtt")
    print("Done extraction.")


Done.
Done.
Done.
Extracting videos...
Done extraction.


## Step 4: Define Encoders and Preprocessing Helpers
We load CLIP vision (student visual), VGGish (student audio), and ImageBind (teacher video).


In [4]:
import clip
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import imagebind.data as ib_data

# 1. CLIP Vision
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
clip_model.eval()

# 2. VGGish via torchhub
vggish = torch.hub.load('harritaylor/torchvggish', 'vggish', trust_repo=True)
vggish.eval()
vggish.to(device)

# 3. ImageBind Video (Teacher)
ib_model = imagebind_model.imagebind_huge(pretrained=True)
ib_model.eval()
ib_model.to(device)

print("Encoders loaded successfully!")


100%|███████████████████████████████████████| 338M/338M [00:03<00:00, 97.2MiB/s]


Downloading: "https://github.com/harritaylor/torchvggish/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth" to /root/.cache/torch/hub/checkpoints/vggish-10086976.pth


100%|██████████| 275M/275M [00:01<00:00, 281MB/s] 


Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish_pca_params-970ea276.pth" to /root/.cache/torch/hub/checkpoints/vggish_pca_params-970ea276.pth


100%|██████████| 177k/177k [00:00<00:00, 7.65MB/s]


100%|██████████| 4.47G/4.47G [01:34<00:00, 50.9MB/s]


Encoders loaded successfully!


## Step 5: Define Extraction Helper Functions
We write robust helpers to extract visual frame, audio waveform, and teacher video embeddings.


In [6]:
def extract_clip_frame(video_path):
    # Reads the middle frame and passes it through CLIP Vision
    vr = VideoReader(video_path, ctx=cpu(0))
    mid_idx = len(vr) // 2
    frame = vr[mid_idx].asnumpy() # [H, W, C]
    pil_img = Image.fromarray(frame)
    img_tensor = clip_preprocess(pil_img).unsqueeze(0).to(device)
    with torch.no_grad():
        feat = clip_model.encode_image(img_tensor)
    return feat.squeeze(0).cpu()

def extract_vggish_audio(video_path):
    # Extracts audio from video path and computes 128-dim VGGish embedding
    import subprocess
    temp_wav = "temp_audio.wav"
    if os.path.exists(temp_wav):
        os.remove(temp_wav)
    # Resample to 16kHz mono using ffmpeg
    cmd = f"ffmpeg -y -i {video_path} -vn -acodec pcm_s16le -ar 16000 -ac 1 {temp_wav}"
    subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    
    # If no audio track, return zero vector
    if not os.path.exists(temp_wav) or os.path.getsize(temp_wav) < 1000:
        if os.path.exists(temp_wav): os.remove(temp_wav)
        return torch.zeros(128)
        
    try:
        with torch.no_grad():
            feat = vggish.forward(temp_wav)
            if feat.ndim > 1:
                feat = feat.mean(dim=0) # Aggregate across frames
        os.remove(temp_wav)
        return feat.cpu()
    except Exception as e:
        if os.path.exists(temp_wav): os.remove(temp_wav)
        return torch.zeros(128)

def extract_imagebind_video(video_path):
    # Passes the full video through ImageBind video pipeline
    inputs = {
        ModalityType.VISION: ib_data.load_and_transform_video_data([video_path], device)
    }
    with torch.no_grad():
        embeddings = ib_model(inputs)
        feat = embeddings[ModalityType.VISION]
    return feat.squeeze(0).cpu()


## Step 6: Feature Extraction Loop
We extract features for both training and testing datasets. Since this takes time, we save progress incrementally.


In [7]:
def run_extraction(split_json, output_pt):
    with open(split_json) as f:
        data = json.load(f)
    
    # Limit to unique videos (MSR-VTT contains multiple sentences per video)
    unique_videos = {}
    for item in data:
        unique_videos[item['video_id']] = item['video']
        
    video_ids = sorted(list(unique_videos.keys()))
    print(f"Extracting features for {len(video_ids)} videos from {split_json}...")
    
    z_imgs = []
    z_auds = []
    v_teachers = []
    valid_video_ids = []
    
    for vid in tqdm(video_ids):
        video_file = os.path.join("msrvtt/MSRVTT_Videos", unique_videos[vid])
        if not os.path.exists(video_file):
            continue
        
        try:
            # Extract visual
            z_img = extract_clip_frame(video_file)
            # Extract audio
            z_aud = extract_vggish_audio(video_file)
            # Extract teacher video
            v_teacher = extract_imagebind_video(video_file)
            
            z_imgs.append(z_img)
            z_auds.append(z_aud)
            v_teachers.append(v_teacher)
            valid_video_ids.append(vid)
        except Exception as e:
            print(f"Error processing {vid}: {e}")
            continue
            
    # Convert to tensors
    out_dict = {
        'z_img': torch.stack(z_imgs),
        'z_aud': torch.stack(z_auds),
        'v_teacher': torch.stack(v_teachers),
        'video_ids': valid_video_ids
    }
    torch.save(out_dict, output_pt)
    print(f"Saved to {output_pt}. Shape z_img: {out_dict['z_img'].shape}, z_aud: {out_dict['z_aud'].shape}, v_teacher: {out_dict['v_teacher'].shape}")


## Step 7: Run extraction
Let's extract features for both training and test sets.


In [8]:
run_extraction("msrvtt/msrvtt_test_1k.json", "test_features.pt")
run_extraction("msrvtt/msrvtt_train_7k.json", "train_features.pt")


Extracting features for 1000 videos from msrvtt/msrvtt_test_1k.json...


100%|██████████| 1000/1000 [00:00<00:00, 221569.15it/s]


RuntimeError: stack expects a non-empty TensorList